# AML Fraud Detection — EDA & Machine Learning
**Dataset:** PaySim — 6.3M synthetic mobile money transactions  
**Goal:** Identify behavioral patterns that distinguish fraudulent transactions from legitimate ones, then build a classification model.

---

## 1. Load Data

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- Load dataset ---
# Option A: file picker dialog (desktop Jupyter)
try:
    from tkinter import filedialog, Tk
    root = Tk()
    root.withdraw()
    file_path = filedialog.askopenfilename(
        title='Select PaySim dataset',
        filetypes=[('CSV files', '*.csv')]
    )
    df = pd.read_csv(file_path)
except Exception:
    # Option B: place the CSV in the same folder and rename it
    df = pd.read_csv('paysim_dataset.csv')

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head()

## 2. Basic Exploration

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_rate = fraud_counts[1] / len(df) * 100
print(f"Legitimate transactions: {fraud_counts[0]:,}")
print(f"Fraudulent transactions:  {fraud_counts[1]:,}")
print(f"Fraud rate: {fraud_rate:.4f}%  → Highly imbalanced dataset")

In [ ]:
print("Average transaction amount by fraud label:")
print(df.groupby('isFraud')['amount'].mean().rename({0: 'Legitimate', 1: 'Fraudulent'}))

## 3. SQL-Based Rule Analysis

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('transactions', conn, if_exists='replace', index=False)

# Balance behavior: fraudulent senders drain their account completely
query = """
SELECT 
    isFraud,
    AVG(oldbalanceOrg)  AS avg_old_balance,
    AVG(newbalanceOrig) AS avg_new_balance
FROM transactions
GROUP BY isFraud
"""
print("Average sender balance before/after transaction:")
print(pd.read_sql_query(query, conn).rename(columns={'isFraud': 'Label'}).to_string(index=False))

In [ ]:
# SQL rule: Finding accounts that committed more than one fraudulent transaction
query_structuring = """
SELECT 
    nameOrig,
    COUNT(*)     AS transaction_count,
    AVG(amount)  AS avg_amount
FROM transactions
WHERE isFraud = 1
GROUP BY nameOrig
HAVING transaction_count > 1
ORDER BY transaction_count DESC
"""
result_structuring = pd.read_sql_query(query_structuring, conn)
print(f"Accounts with multiple fraud transactions: {len(result_structuring)}")
print(result_structuring.head(10))

In [ ]:
# High-value transactions are NOT necessarily fraudulent — rule-based thresholds create false positives
query_flags = """
SELECT type, amount, isFraud
FROM transactions
WHERE amount > 1000000
ORDER BY amount DESC
LIMIT 10
"""
print("Top 10 largest transactions:")
print(pd.read_sql_query(query_flags, conn).to_string(index=False))

## 4. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('AML Fraud Detection — Exploratory Analysis', fontsize=16, fontweight='bold', y=1.01)

COLORS = {'fraud': '#e74c3c', 'legit': '#2ecc71', 'neutral': '#3498db'}

# --- Plot 1: Fraud distribution (pie) ---
ax1 = axes[0, 0]
labels = ['Legitimate\n(99.87%)', 'Fraudulent\n(0.13%)']
sizes  = [fraud_counts[0], fraud_counts[1]]
colors = [COLORS['legit'], COLORS['fraud']]
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.2f%%',
        startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('Class Distribution', fontweight='bold')

# --- Plot 2: Fraud by transaction type ---
ax2 = axes[0, 1]
fraud_by_type = df[df['isFraud'] == 1]['type'].value_counts()
bars = ax2.bar(fraud_by_type.index, fraud_by_type.values, color=COLORS['fraud'], edgecolor='white', linewidth=0.8)
ax2.set_title('Fraudulent Transactions by Type', fontweight='bold')
ax2.set_xlabel('Transaction Type')
ax2.set_ylabel('Count')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
             f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)

# --- Plot 3: Amount distribution (log scale) ---
ax3 = axes[1, 0]
legit_amounts = np.log1p(df[df['isFraud'] == 0]['amount'].sample(20000, random_state=42))
fraud_amounts = np.log1p(df[df['isFraud'] == 1]['amount'])
ax3.hist(legit_amounts, bins=60, alpha=0.6, color=COLORS['legit'], label='Legitimate', density=True)
ax3.hist(fraud_amounts, bins=60, alpha=0.7, color=COLORS['fraud'], label='Fraudulent', density=True)
ax3.set_title('Transaction Amount Distribution (log scale)', fontweight='bold')
ax3.set_xlabel('log(1 + amount)')
ax3.set_ylabel('Density')
ax3.legend()

# --- Plot 4: Balance drain pattern ---
ax4 = axes[1, 1]
categories  = ['Avg Balance Before', 'Avg Balance After']
legit_vals  = [df[df['isFraud']==0]['oldbalanceOrg'].mean(),  df[df['isFraud']==0]['newbalanceOrig'].mean()]
fraud_vals  = [df[df['isFraud']==1]['oldbalanceOrg'].mean(),  df[df['isFraud']==1]['newbalanceOrig'].mean()]
x = np.arange(len(categories))
w = 0.35
ax4.bar(x - w/2, legit_vals, w, label='Legitimate', color=COLORS['legit'], edgecolor='white')
ax4.bar(x + w/2, fraud_vals, w, label='Fraudulent', color=COLORS['fraud'], edgecolor='white')
ax4.set_title('Sender Balance: Before vs After Transaction', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(categories)
ax4.set_ylabel('Average Amount ($)')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax4.legend()

plt.tight_layout()
plt.savefig('aml_eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved to aml_eda_charts.png")

## 5. Feature Engineering

In [ ]:
# Keep only transaction types where fraud actually occurs
df_model = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# New features based on EDA findings
df_model['balance_drained']      = (df_model['newbalanceOrig'] == 0).astype(int)
df_model['dest_balance_no_change'] = (df_model['newbalanceDest'] == df_model['oldbalanceDest']).astype(int)
df_model['amount_to_orig_ratio'] = df_model['amount'] / (df_model['oldbalanceOrg'] + 1)
df_model['type_encoded']         = (df_model['type'] == 'CASH_OUT').astype(int)  # 1=CASH_OUT, 0=TRANSFER

print(f"Filtered dataset: {len(df_model):,} rows")
print(f"Fraud rate in filtered set: {df_model['isFraud'].mean()*100:.3f}%")

## 6. Machine Learning Model

> **Note on class imbalance:** Fraud is only ~0.3% of the TRANSFER/CASH_OUT subset.  
> We use `class_weight='balanced'` to penalize misclassifying the minority class, and evaluate with **Precision, Recall and F1** rather than accuracy.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

FEATURES = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'balance_drained', 'dest_balance_no_change',
    'amount_to_orig_ratio', 'type_encoded'
]

X = df_model[FEATURES]
y = df_model['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Fraud in train: {y_train.sum():,}  |  Fraud in test: {y_test.sum():,}")

# Random Forest with class_weight to handle imbalance
clf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

# Confusion Matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Legitimate', 'Fraud']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# Feature Importance
importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('aml_model_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Findings

| Finding | Detail |
|---|---|
| **Fraud is type-specific** | 100% of fraud occurs in TRANSFER and CASH_OUT transactions only |
| **Balance draining** | Fraudulent senders deplete their entire balance; avg drops from ~$1.65M to ~$192K |
| **Amount ≠ Fraud** | Top 10 largest transactions (>$60M) are all legitimate — simple thresholds cause high false positives |
| **No repeat offenders** | Each fraudulent account appears exactly once — no structuring pattern detected |
| **Model performance** | Random Forest with engineered features achieves strong ROC-AUC; `balance_drained` and `amount_to_orig_ratio` are top predictors |

---
*Dataset: PaySim synthetic financial dataset | Tools: Python, Pandas, SQLite, Scikit-learn, Matplotlib*